In [1]:
import torch
from torch_geometric.datasets import QM9
from rdkit import Chem
from rdkit import RDLogger
import pandas as pd
import os

# Suppress RDKit valence and sanitization warnings
RDLogger.DisableLog('rdApp.*')

In [2]:
# ------------------------------------------------------------
# Load QM9 Dataset
# QM9 contains 130k small organic molecules with DFT properties
# ------------------------------------------------------------
dataset = QM9(root='data/')

Extracting data/raw/qm9.zip
Processing...
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 133885/133885 [01:51<00:00, 1201.99it/s]
Done!


In [10]:
# QM9 target indices:
# 0: mu, 1: alpha, 2: homo, 3: lumo, 4: gap, 5: 
HOMO_IDX = 2   # HOMO energy (eV)
LUMO_IDX = 3   # LUMO energy (eV)
GAP_IDX  = 4   # HOMO-LUMO gap / bandgap (eV)

# ------------------------------------------------------------
# Quinone SMARTS Pattern
# Matches the core C(=O)-C=C-C(=O) motif shared by all quinones
# ------------------------------------------------------------
quinone_smarts = Chem.MolFromSmarts('C(=O)C=CC(=O)')

# Storage for matched molecules
quinone_indices = []   # Position in QM9 dataset
quinone_smiles  = []   # SMILES string
quinone_names   = []   # GDB-17 name (e.g. gdb_12345)
quinone_ids     = []   # Official GDB-17 index
quinone_homo    = []   # HOMO energy (eV)
quinone_lumo    = []   # LUMO energy (eV)
quinone_gap     = []   # Bandgap / HOMO-LUMO gap (eV)

# ------------------------------------------------------------
# Scan QM9 for Quinone Derivatives
# ------------------------------------------------------------
print("Searching QM9 for quinone derivatives...")
print(f"Using SMARTS pattern: C(=O)C=CC(=O)\n")

for i, data in enumerate(dataset):
    if i % 10000 == 0:
        print(f"  Scanned {i}/{len(dataset)}...")

    mol = Chem.MolFromSmiles(data.smiles)
    if mol is None:
        continue

    if mol.HasSubstructMatch(quinone_smarts):
        quinone_indices.append(i)
        quinone_smiles.append(data.smiles)
        quinone_names.append(data.name)
        quinone_ids.append(data.idx.item())
        quinone_homo.append(data.y[0, HOMO_IDX].item())
        quinone_lumo.append(data.y[0, LUMO_IDX].item())
        quinone_gap.append(data.y[0, GAP_IDX].item())

        # Stop once we have 30 quinone samples
        if len(quinone_indices) == 30:
            break

# ------------------------------------------------------------
# Confirm These Are Quinones
# ------------------------------------------------------------
print(f"\nFound {len(quinone_indices)} quinone derivatives in QM9")
print("\nVerifying quinone substructure in all collected molecules...")

verified = 0
for smi in quinone_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol and mol.HasSubstructMatch(quinone_smarts):
        verified += 1

print(f"Verified: {verified}/{len(quinone_smiles)} molecules confirmed as quinones")
if verified == len(quinone_smiles):
    print("All molecules contain the quinone core C(=O)-C=C-C(=O)\n")
else:
    print(f"{len(quinone_smiles) - verified} molecules failed verification\n")

# ------------------------------------------------------------
# Build and Save DataFrame
# ------------------------------------------------------------
df = pd.DataFrame({
    'gdb_name'        : quinone_names,
    'gdb_idx'         : quinone_ids,
    'qm9_dataset_pos' : quinone_indices,
    'smiles'          : quinone_smiles,
    'homo_eV'         : quinone_homo,
    'lumo_eV'         : quinone_lumo,
    'bandgap_eV'      : quinone_gap,
})

df.to_csv('quinone_molecules.csv', index=False)
print("Saved to quinone_molecules.csv\n")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print(f"Dataset summary:")
print(f"  Total quinone molecules : {len(df)}")
print(f"  HOMO range  : {df['homo_eV'].min():.3f} – {df['homo_eV'].max():.3f} eV")
print(f"  LUMO range  : {df['lumo_eV'].min():.3f} – {df['lumo_eV'].max():.3f} eV")
print(f"  Gap range   : {df['bandgap_eV'].min():.3f} – {df['bandgap_eV'].max():.3f} eV")
print(f"\n{df.to_string()}")

Searching QM9 for quinone derivatives...
Using SMARTS pattern: C(=O)C=CC(=O)

  Scanned 0/130831...
  Scanned 10000/130831...
  Scanned 20000/130831...
  Scanned 30000/130831...
  Scanned 40000/130831...

Found 30 quinone derivatives in QM9

Verifying quinone substructure in all collected molecules...
Verified: 30/30 molecules confirmed as quinones
All molecules contain the quinone core C(=O)-C=C-C(=O)

Saved to quinone_molecules.csv

Dataset summary:
  Total quinone molecules : 30
  HOMO range  : -8.008 – -6.055 eV
  LUMO range  : -3.535 – -1.804 eV
  Gap range   : 3.543 – 5.328 eV

     gdb_name  gdb_idx  qm9_dataset_pos                                                  smiles   homo_eV   lumo_eV  bandgap_eV
0    gdb_1821     1820             1768                        [H]C1=C([H])C(=O)C([H])([H])C1=O -6.851827 -2.623178    4.231370
1    gdb_1822     1821             1769                             [H]C1=C([H])C(=O)N([H])C1=O -7.385170 -2.625899    4.759271
2    gdb_7875     7874   